# LineSight AI — YOLO Dataset v1 / czjyz_ps

Только подготовка и проверка датасета. Цель — локальное повреждение керамики, а не целый изолятор. Обучение и GPU-инференс не запускаются.

Используйте свежую Colab CPU VM. T4 потребуется позже, после согласования эксперимента. Подготовьте `colab_dataset_v1_bundle.zip` из локальной папки `ml/runs/czjyz-ps-audit-v1/`. Пакет содержит точные split, метки и версию скриптов; публикация приватного кода в GitHub не нужна.

Выборка содержит только целевые позитивные изображения. Проверенные негативные кадры пока отсутствуют. Только 48 изображений прошли выборочный семантический аудит; независимость соседних ракурсов не гарантируется эвристическими хешами.


In [ ]:
from pathlib import Path
import hashlib, json, sys, zipfile, subprocess
from google.colab import files
EXPECTED_BUNDLE_SHA256 = "b1415dc7d026653cbeb968649ee5456f477557c44cfe0940149e0529ab0001d4"
uploaded = files.upload()  # выбрать только colab_dataset_v1_bundle.zip
assert len(uploaded) == 1
bundle_name, bundle_bytes = next(iter(uploaded.items()))
assert hashlib.sha256(bundle_bytes).hexdigest() == EXPECTED_BUNDLE_SHA256, 'Wrong or changed bundle'
work = Path('/content/linesight_v1')
work.mkdir(exist_ok=False)
with zipfile.ZipFile(bundle_name) as archive:
    allowed = {'prepare_yolo.py','download_dataset.py','check_env.py','voc_dataset.py','plan.json','bundle_manifest.json'}
    assert len(archive.infolist()) == len(allowed) and set(archive.namelist()) == allowed
    assert sum(i.file_size for i in archive.infolist()) < 30 * 1024**2
    metadata = json.loads(archive.read('bundle_manifest.json'))
    for name in allowed:
        data = archive.read(name)
        if name != 'bundle_manifest.json':
            assert hashlib.sha256(data).hexdigest() == metadata['files'][name]
        (work / name).write_bytes(data)
del uploaded, bundle_bytes
sys.path.insert(0, str(work))
plan = json.loads((work/'plan.json').read_text())
print(plan['split_counts'], plan['split_objects'])


Источник данных: `upload` использует уже скачанный ZIP; `hf` адресно скачивает один архив с Hugging Face. Измените `SOURCE_MODE` при необходимости. В режиме HF revision `main` допустим только вместе с последующей строгой проверкой SHA256: если файл изменился, работа остановится. Токены в код не вставлять; для этого публичного архива они обычно не нужны.


In [ ]:
SOURCE_MODE = 'upload'
if SOURCE_MODE == 'upload':
    uploaded_archive = files.upload()  # выбрать transmission line object detection.zip
    assert len(uploaded_archive) == 1
    archive_path = Path('/content') / next(iter(uploaded_archive))
    del uploaded_archive
elif SOURCE_MODE == 'hf':
    import importlib.util
    if importlib.util.find_spec('huggingface_hub') is None:
        subprocess.run([sys.executable,'-m','pip','install','huggingface_hub>=0.24,<2'],check=True)
    result = subprocess.run([sys.executable,str(work/'download_dataset.py'),'--download','--revision','main','--cache-dir','/content/hf-cache'],check=True,capture_output=True,text=True)
    archive_path = Path(json.loads(result.stdout)['archive'])
else:
    raise ValueError('Choose upload or hf')
from download_dataset import inspect_archive, safe_extract
archive_info = inspect_archive(archive_path)
assert archive_info['sha256'] == plan['source_archive']['sha256'], 'Archive differs from reviewed source; stop'
print('Archive verified:', archive_info['size_bytes'], 'bytes; unpacked:', archive_info['uncompressed_bytes'])
source = safe_extract(archive_path, '/content/source_v1')


In [ ]:
dataset = Path('/content/yolo_czjyz_ps_v1')
subprocess.run([sys.executable,str(work/'prepare_yolo.py'),'export','--plan',str(work/'plan.json'),'--source',str(source),'--output',str(dataset),'--hardlink'],check=True)
result = subprocess.run([sys.executable,str(work/'prepare_yolo.py'),'validate','--dataset',str(dataset)],check=True,capture_output=True,text=True)
print(result.stdout)
from check_env import check_environment
(work/'environment.json').write_text(json.dumps(check_environment(),indent=2))
(work/'validation.json').write_text(result.stdout)
print('Dataset YAML:', dataset/'data.yaml')
print('STOP: dataset prepared; training has not been approved or started.')


In [ ]:
receipt = Path('/content/linesight_dataset_v1_receipt.zip')
with zipfile.ZipFile(receipt,'x',compression=zipfile.ZIP_DEFLATED) as out:
    out.write(work/'environment.json','environment.json')
    out.write(work/'validation.json','validation.json')
    out.write(dataset/'manifest.json','manifest.json')
    out.write(dataset/'data.yaml','data.yaml')
files.download(str(receipt))


## На этом остановиться

В ноутбуке нет вызовов обучения, скачивания весов или инференса. Сравнение 640/960 на неизменном split — возможный следующий согласованный эксперимент с небольшой одноступенчатой моделью YOLO.

Изображения размещены через hard links: не редактируйте их на месте. Исходные VOC и ZIP сохраняются. При сбросе Colab `/content` исчезает; скачайте ZIP отчёта на компьютер и положите его в ignored `ml/runs/`. Drive не обязателен.

Скрипты зафиксированы SHA256 в пакете. Версии реально использованных пакетов сохраняются в `environment.json`. Этот notebook проверен локально структурно; выполнение через Colab UI ещё не подтверждено.
